# BrainTumorAI Google Colab Backend
This notebook runs the BrainTumorAI FastAPI server directly on Google Colab's GPU.
**Instructions:**
1. Ensure you are using a **T4 GPU** runtime (Runtime -> Change runtime type -> Hardware accelerator: T4 GPU).
2. Press **Run all** (Cmd/Ctrl + F9). The notebook will automatically clone the codebase, setup models, and start the API.

In [ ]:
import os
import subprocess

# Clone the GitHub repository automatically if it does not exist
if not os.path.exists('/content/BrainTumorAI'):
    print("Cloning repository...")
    subprocess.run(["git", "clone", "https://github.com/BiswasApurbo/BrainTumorAI.git", "/content/BrainTumorAI"], check=True)
else:
    print("Repository already exists. Pulling latest changes...")
    subprocess.run(["git", "pull"], cwd="/content/BrainTumorAI", check=True)

%cd /content/BrainTumorAI


In [ ]:
import os
import sys
import subprocess

print(f"Python Version: {sys.version.split()[0]}")

print("Installing dependencies (uv)...")
subprocess.run("curl -LsSf https://astral.sh/uv/install.sh | sh", shell=True, check=True, stdout=subprocess.DEVNULL)
os.environ["PATH"] += ":/root/.cargo/bin"

print("Syncing project dependencies...")
subprocess.run(["uv", "sync"], check=True, stdout=subprocess.DEVNULL)

# Verify PyTorch and CUDA inside the uv environment
verify_script = """
import torch
print(f'PyTorch Version: {torch.__version__}')
cuda_available = torch.cuda.is_available()
print(f'CUDA Available: {cuda_available}')
if cuda_available:
    print(f'GPU Model: {torch.cuda.get_device_name(0)}')
else:
    import sys
    sys.exit(1)
"""

try:
    subprocess.run(["uv", "run", "python", "-c", verify_script], check=True)
except subprocess.CalledProcessError:
    raise RuntimeError("CUDA is not available. Please ensure you are using a GPU runtime (T4).")


In [ ]:
import os

local_models_path = '/content/BrainTumorAI/models'

# 1. Verify nnUNet
nnunet_checkpoint = os.path.join(
    local_models_path,
    'nnUNet/3d_fullres/Task082_BraTS2020/nnUNetTrainerV2BraTSRegions_DA4_BN__nnUNetPlansv2.1_bs5/fold_0/model_final_checkpoint.model'
)

if not os.path.exists(nnunet_checkpoint):
    raise FileNotFoundError(f"nnUNet checkpoint missing at {nnunet_checkpoint}")
    
nnunet_size = os.path.getsize(nnunet_checkpoint)
if nnunet_size < 100 * 1024 * 1024:  # Less than 100MB indicates an LFS stub
    raise ValueError(f"nnUNet checkpoint seems too small ({nnunet_size} bytes). Real weights required.")
    
if not os.access(nnunet_checkpoint, os.R_OK):
    raise PermissionError(f"nnUNet checkpoint at {nnunet_checkpoint} is not readable.")

# 2. Verify SynthSeg
synthseg_checkpoint = os.path.join(
    local_models_path,
    'SynthSeg/models/synthseg_1.0.h5'
)

if not os.path.exists(synthseg_checkpoint):
    raise FileNotFoundError(f"SynthSeg checkpoint missing at {synthseg_checkpoint}")
    
synthseg_size = os.path.getsize(synthseg_checkpoint)
if synthseg_size < 10 * 1024 * 1024: # Less than 10MB indicates an LFS stub
    raise ValueError(f"SynthSeg checkpoint seems too small ({synthseg_size} bytes). Real weights required.")

if not os.access(synthseg_checkpoint, os.R_OK):
    raise PermissionError(f"SynthSeg checkpoint at {synthseg_checkpoint} is not readable.")

print(f"Verified nnUNet checkpoint: {nnunet_size / (1024*1024):.1f} MB")
print(f"Verified SynthSeg checkpoint: {synthseg_size / (1024*1024):.1f} MB")

# 3. Environment Variables
os.environ['RESULTS_FOLDER'] = local_models_path
os.environ['nnUNet_raw_data_base'] = os.path.join(local_models_path, 'nnUNet_raw_data_base')
os.environ['nnUNet_preprocessed'] = os.path.join(local_models_path, 'nnUNet_preprocessed')

print("\nConfigured Environment Variables:")
print(f"RESULTS_FOLDER={os.environ['RESULTS_FOLDER']}")
print(f"nnUNet_raw_data_base={os.environ['nnUNet_raw_data_base']}")
print(f"nnUNet_preprocessed={os.environ['nnUNet_preprocessed']}")


In [ ]:
import subprocess
import time
import requests
import os

print("Starting FastAPI backend...")

env = dict(os.environ)
env["PYTHONPATH"] = "src"

# Start uvicorn in the background
server_process = subprocess.Popen(
    ["uv", "run", "uvicorn", "backend.main:app", "--host", "0.0.0.0", "--port", "8000"],
    env=env,
    stdout=subprocess.DEVNULL,
    stderr=subprocess.STDOUT
)

print("Polling /health until backend is ready (timeout 60s)...")
healthy = False
start_time = time.time()
while time.time() - start_time < 60:
    try:
        response = requests.get("http://127.0.0.1:8000/health")
        if response.status_code == 200:
            healthy = True
            break
    except requests.ConnectionError:
        pass
    time.sleep(1)

if not healthy:
    server_process.terminate()
    raise RuntimeError("FastAPI backend failed to start or did not become healthy within 60 seconds.")

print("FastAPI is healthy and running!")


In [ ]:
!pip install pyngrok -q
from pyngrok import ngrok
import time
import requests
import torch

public_url = ngrok.connect(8000).public_url
local_url = "http://127.0.0.1:8000"

print("\n====================================")
print("         STARTUP SUMMARY            ")
print("====================================")
print(f"✓ CUDA detected")
print(f"✓ GPU model: {torch.cuda.get_device_name(0)}")
print(f"✓ Repository path: /content/BrainTumorAI")
print(f"✓ nnUNet checkpoint found")
print(f"✓ SynthSeg checkpoint found")
print(f"✓ FastAPI healthy")
print(f"✓ ngrok running")
print("====================================\n")

print(f"Local URL:  {local_url}")
print(f"Public URL: {public_url}\n")
print(f">>> OPEN THIS URL IN YOUR BROWSER: {public_url} <<<\n")

print("Verifying API externally (lightweight verification)...")
try:
    res = requests.get(f"http://127.0.0.1:8000/health")
    if res.status_code == 200:
        print("✓ GET /health returns 200 OK.")
    else:
        print(f"✗ GET /health returned {res.status_code}")
except Exception as e:
    print(f"✗ Failed to connect to health endpoint: {e}")

# Keep the cell running to prevent the tunnel and server from dying
try:
    while True:
        time.sleep(1)
except KeyboardInterrupt:
    print("Shutting down...")
    server_process.terminate()
    ngrok.kill()
